# Audible — Self-Improving Ambient-Listening Gate

Hackathon submission notebook for the **OpenEnv Hackathon (India 2026), Theme #4 — Self-Improvement**.

This notebook trains a tiny **mobileBERT** classifier (~25M params, edge-deployable) to act as the
**gate** of an always-on voice assistant: given an ambient utterance and the active user's preference
profile, decide whether to ACT (and which of 5 tools to call), UPDATE_CONTEXT, or IGNORE. It then
evaluates the trained model against our OpenEnv environment and shows reward + per-component metrics.

**Run all → ~10–15 min on a free Colab T4**.

Repo: https://github.com/me-tusharchandra/audible  •  Env Space: https://huggingface.co/spaces/me-tusharchandra/audible-env

## 1. Install dependencies

In [ ]:
%%capture
# OpenEnv core (env framework) + ML stack
!pip install --quiet "openenv-core[core] @ git+https://github.com/meta-pytorch/OpenEnv.git"
!pip install --quiet transformers torch accelerate datasets scikit-learn matplotlib pandas pyarrow python-dotenv openai

## 2. Pull the project repo

The notebook trains against the `combined.parquet` dataset that lives in this repo, then evaluates against the audible env (started locally inside the Colab VM).

In [ ]:
import os, subprocess, sys, time, pathlib
REPO_DIR = pathlib.Path('/content/audible')
if not REPO_DIR.exists():
    !git clone --depth 1 https://github.com/me-tusharchandra/audible.git {REPO_DIR}
%cd {REPO_DIR}
sys.path.insert(0, str(REPO_DIR))
print('repo at:', REPO_DIR)

## 3. Configure secrets (optional — only needed for synthetic data + curriculum)

In [ ]:
from google.colab import userdata  # comment out if running locally
try:
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print('OPENAI_API_KEY set — synthetic generation + curriculum available')
except Exception as e:
    print('No OPENAI_API_KEY in Colab Secrets — skipping synth/curriculum (eval still works)')

## 4. Inspect the dataset

Combined dataset = friend's binary CSV (heuristic-mapped to 7 classes) + LLM-generated synthetic
scenarios that target ambient/profile-divergent cases the binary data lacks. ~7.5k unique utterances
× 3 user profiles = ~22k labeled rows.

In [ ]:
import pandas as pd
df = pd.read_parquet('audible_env/data/combined.parquet')
print(f'rows: {len(df):,}   utterances: {df["text"].nunique():,}')
print('\nclass distribution per profile:')
print(df.pivot_table(index='class_label', columns='profile', values='text', aggfunc='count', fill_value=0).to_string())

## 5. Train mobileBERT (gating classifier)

1 epoch on the combined dataset takes ~3 min on a Colab T4. The training script is a custom
`WeightedTrainer` (HF `Trainer` subclass) with class-weighted CE loss to compensate for the
natural class imbalance. Outputs go to `training/runs/<timestamp>/`.

> *Note on TRL*: TRL's `PPOTrainer`/`GRPOTrainer` are built for generative LMs and don't fit
> encoder-only classifiers. Our `WeightedTrainer` is a `transformers.Trainer` subclass — the same
> base class TRL extends, with class-weighted CE in `compute_loss`.

In [ ]:
!python -m training.train --epochs 1 --batch 32 2>&1 | tail -30

## 6. Start the OpenEnv server (locally inside this VM)

In [ ]:
import subprocess, time, urllib.request
ENV_PORT = 8765
env_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'server.app:app',
     '--host', '127.0.0.1', '--port', str(ENV_PORT)],
    cwd='audible_env', stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
for _ in range(30):
    try:
        urllib.request.urlopen(f'http://127.0.0.1:{ENV_PORT}/health', timeout=1).read()
        print('env server up'); break
    except Exception:
        time.sleep(0.5)
else:
    raise RuntimeError('env server did not start')

## 7. Evaluate the trained gate against the env

Runs ~400 rollouts: env serves an utterance + profile, the gate classifies, env returns reward
via the composite rubric (gate correctness + tool match + profile alignment + false-wake penalty).
Aggregates per-profile so we can see how each preference profile is performing separately.

In [ ]:
import json, pathlib
from training.eval_env import GatePolicy, run_rollouts, summarize

runs = sorted(pathlib.Path('training/runs').glob('*'))
model_dir = runs[-1]
print('using model:', model_dir.name)

policy = GatePolicy(model_dir)
rollouts = run_rollouts(policy, base_url=f'http://127.0.0.1:{ENV_PORT}', n=400, seed=0)
summary = summarize(rollouts)
print(json.dumps(summary, indent=2))

## 8. Visualise per-profile metrics

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

profiles = ['minimalist', 'proactive', 'work_focused']
metrics_to_plot = ['mean_reward', 'decision_accuracy', 'tool_accuracy_when_act', 'false_wake_rate']
vals = np.array([[summary['per_profile'][p][m] for m in metrics_to_plot] for p in profiles])

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(metrics_to_plot)); w = 0.25
for i, p in enumerate(profiles):
    ax.bar(x + i * w, vals[i], w, label=p)
ax.set_xticks(x + w); ax.set_xticklabels(metrics_to_plot, rotation=15)
ax.legend(); ax.set_title('Per-profile env-rollout metrics')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

## 9. Theme #4: adaptive curriculum (optional, requires OPENAI_API_KEY)

After each training round we:
1. Eval the current gate against the env
2. Mine the worst-reward rollouts as adversarial seeds
3. Generate new scenarios via OpenAI: *"here are utterances the gate gets wrong — make harder variations"*
4. Append to the dataset and retrain

Across rounds the reward curve climbs because the gate gets stronger; the generator keeps escalating
because it sees the gate's current weak spots. **That co-evolution is what makes this self-improvement.**

Skipped here (each round adds ~10 min of compute). To run locally:
```bash
python -m training.curriculum --rounds 3 --gen-per-round 200
```
Pre-computed curriculum results are in `training/plots/reward_curve.png`.

## 10. Cleanup

In [ ]:
env_proc.terminate(); env_proc.wait(timeout=5); print('env server stopped')